```{index} single: Pyomo; variables

```

```{index} single: Pyomo; expressions

```

```{index} single: Pyomo; sets

```

```{index} single: Pyomo; decorators

```

```{index} single: Pyomo; parameters

```

```{index} single: solver; HiGHS

```

# 1.3 Un modelo Pyomo basado en datos

En este cuaderno, revisaremos el ejemplo de planificación de producción. Sin embargo, esta vez demostraremos cómo las estructuras de datos de Python, combinadas con las capacidades de Pyomo, pueden crear un modelo de optimización que se adapta a los datos del problema. Esto permite que el modelo se ajuste a nuevos productos, precios variables o demanda cambiante. Nos referimos a esto como modelado "basado en datos".

Los componentes adicionales de Pyomo utilizados en este cuaderno son:

- [Conjuntos](https://pyomo.readthedocs.io/en/6.8.0/pyomo_modeling_components/Sets.html)
- [Parámetros](https://pyomo.readthedocs.io/en/6.8.0/pyomo_modeling_components/Parameters.html)

Estos componentes permiten el uso de variables y restricciones indexadas. La combinación de conjuntos e índices es esencial para crear modelos escalables y mantenibles para aplicaciones más complejas.

Comenzaremos este análisis examinando los conjuntos de datos del problema para identificar la estructura subyacente del problema.

## Preámbulo: instalar Pyomo y un solucionador

La siguiente celda verifica si el cuaderno se ejecuta en Google Colab. Si es así, realiza una instalación silenciosa de Pyomo y el solucionador HiGHS. Luego se selecciona el solucionador, se realiza una prueba para verificar que esté disponible y la interfaz del solucionador se almacena en un objeto global `SOLVER` para su uso posterior.

In [1]:
import sys
 
if 'google.colab' in sys.modules:
    %pip install pyomo >/dev/null 2>/dev/null
    %pip install highspy >/dev/null 2>/dev/null
 
solver = 'appsi_highs'
 
import pyomo.environ as pyo
SOLVER = pyo.SolverFactory(solver)

assert SOLVER.available(), f"Solver {solver} is not available."

## Representaciones de datos

Comenzamos revisando los conjuntos de datos y el modelo matemático desarrollados para el problema básico de planificación de la producción presentado en el [cuaderno anterior] (02-production-planning-basic.ipynb). Los conjuntos de datos originales se proporcionaron como:

<div align="center">

| Producto | Material <br> requerido | Se requiere mano de obra A <br> | Se requiere mano de obra B <br> | Mercado <br> Demanda | Precio |
| :-----: | :--------------------: | :------------------: | :------------------: | :----------------: | :---: |
|    U |          10 gramos |         1 hora |         2 horas |  $\leq$ 40 unidades | 270$ |
|    V |          9 gramos |         1 hora |         1 hora |     ilimitado | 210$ |

</div>
y
<div align="center">

| Recurso | Cantidad <br> Disponible |    Costo |
| :------: | :------------------: | :--------: |
| Materiales |       ilimitado |  10$ / gramo |
| Labor A |       80 horas | 50$ / hora |
| Trabajo B |       100 horas | 40$ / hora |

</div>

De estas tablas se desprenden dos conjuntos distintos de objetos. El primero es el conjunto de productos compuesto por $U$ y $V$. El segundo es el conjunto de recursos utilizados para producir esos productos, que abreviamos como $M$, $A$ y $B$.

Una vez identificados estos conjuntos, los datos para esta aplicación se factorizarán en tres tablas simples. Las dos primeras tablas enumeran los atributos de los productos y recursos, la tercera tabla resume el proceso utilizado para crear los productos a partir de los recursos:

**Tabla: Productos**

<div align="center">

| Producto |     Demanda | Precio |
| :-----: | :-------------: | :---: |
|    U | $\leq$ 40 unidades | 270$ |
|    V |    ilimitado | 210$ |

</div>

**Tabla: Recursos**

<div align="center">

| Recurso | Disponible |    Costo |
| :------: | :-------: | :--------: |
| Materiales | ilimitado |  10$ / gramo |
| Labor A | 80 horas | 50$ / hora |
| Trabajo B | 100 horas | 40$ / hora |

</div>

**Tabla: Procesos**

<div align="center">

| Producto | Materiales | Labor A | Trabajo B |
| :-----: | :------: | :-----: | :-----: |
|    U |   10 gramos |  1 hora |  2 horas |
|    V |   9 gramos |  1 hora |  1 hora |

</div>

Python tiene muchos tipos de datos y bibliotecas integrados que son útiles para manejar datos tabulares, y hay varias opciones que serían apropiadas para la tarea en cuestión. Los diccionarios anidados pueden ser una buena opción para problemas más pequeños que tienen sólo unas pocas columnas. En los siguientes ejemplos, mostraremos cómo se pueden usar diccionarios anidados para representar las tres tablas que se describieron anteriormente.

La primera tabla de estos describe los productos. Los nombres de los productos servirán como claves para el diccionario externo y los nombres de atributos como claves para los diccionarios internos. Los valores de los atributos se interpretarán como números de punto flotante. `None` se utiliza cuando un valor no está presente.

In [2]:
products = {
    "U": {"price": 270, "demand": 40},
    "V": {"price": 210, "demand": None},
}

# print data
for product, attributes in products.items():
    for attribute, value in attributes.items():
        print(f"{product} {attribute:10s} {value}")

U price      270
U demand     40
V price      210
V demand     None


La segunda tabla es el diccionario anidado que enumera los atributos y valores de los recursos consumidos.

In [3]:
resources = {
    "M": {"price": 10, "available": None},
    "labor A": {"price": 50, "available": 80},
    "labor B": {"price": 40, "available": 100},
}

for resource, attributes in resources.items():
    for attribute, value in attributes.items():
        print(f"{resource:8s} {attribute:10s} {value}")

M        price      10
M        available  None
labor A  price      50
labor A  available  80
labor B  price      40
labor B  available  100


Los datos de la tercera tabla muestran la cantidad de cada recurso necesario para producir una unidad de cada producto. Las filas están etiquetadas por producto y las columnas por recurso.

In [4]:
processes = {
    "U": {"M": 10, "labor A": 1, "labor B": 2},
    "V": {"M": 9, "labor A": 1, "labor B": 1},
}

for product, process in processes.items():
    for resource, value in process.items():
        print(f"{product:4s} {resource:10s} {value}")

U    M          10
U    labor A    1
U    labor B    2
V    M          9
V    labor A    1
V    labor B    1


## Modelo matemático

Al reorganizar los datos del problema en tablas sencillas, se hace evidente la estructura del problema de planificación de la producción. Podemos identificar un conjunto de productos, un conjunto de recursos y una colección de parámetros que especifican los procesos para transformar recursos en productos. En comparación con el cuaderno anterior, estas abstracciones nos permiten crear modelos matemáticos que pueden adaptarse y escalarse con los datos proporcionados.

Sean $\cal{P}$ y $\cal{R}$ el conjunto de productos y recursos, respectivamente, y sean $p$ y $r$ elementos representativos de esos conjuntos. Usamos las variables de decisión indexadas $x_r$ y $y_p$ para indicar la cantidad de recurso $r$ que se consume en producción, y $y_p$ para indicar la cantidad de producto $p$ producido.

Los datos del problema proporcionan atributos que limitan las variables de decisión. Por ejemplo, todas las variables de decisión tienen límites inferiores de cero y algunas tienen límites superiores. Los representamos como

$$
\begin{aligned}
    0 \leq x_r \leq b_r & & \forall r\in\cal{R} \\
    0 \leq y_p \leq b_p & & \forall p\in\cal{P} \\
\end{aligned}
$$

donde los límites superiores, $b_r$ y $b_p$, provienen de las tablas de atributos. Para los casos en los que los límites superiores no se aplican, podemos insertar límites más grandes de los que jamás se encontrarían o, cuando traduzcamos este modelo a Pyomo, designar un valor especial que haga que se ignore el límite.

El objetivo se da como antes,

$$
\begin{aligned}
    \text{profit} & = \text{revenue} - \text{cost} \\
\end{aligned}
$$

pero ahora las expresiones de ingresos y costos son

$$
\begin{aligned}
    \text{revenue} & = \sum_{p\in\cal{P}} c_p y_p  \\
    \text{cost} & = \sum_{r\in\cal{R}} c_r x_r \\
\end{aligned}
$$

donde $c_r$ y $c_p$ son parámetros que especifican el precio de los recursos y productos. Los límites de los recursos disponibles se pueden escribir como

$$
\begin{aligned}
    \sum_{p\in\cal{P}} a_{r, p} y_p & \leq x_r & \forall r\in\cal{R}
\end{aligned}
$$

Juntando estas piezas, tenemos el siguiente modelo para el problema de planificación de la producción.

$$
\begin{align}
\max \quad & \text{profit} = \sum_{p\in\cal{P}} c_p y_p - \sum_{r\in\cal{R}} c_r x_r \\
\text{such that} \quad & \sum_{p\in\cal{P}} a_{r, p} y_p  \leq x_r & \forall r\in\cal{R} \nonumber \\
 &   0 \leq x_r \leq b_r & \forall r\in\cal{R} \nonumber  \\
 &   0 \leq y_p \leq b_p & \forall p\in\cal{P} \nonumber  \\
\end{align}
$$

En comparación con el cuaderno anterior, cuando se formula de esta manera, el modelo se puede aplicar a cualquier problema con la misma estructura, independientemente de la cantidad de productos o recursos. Esta flexibilidad es posible gracias al uso de conjuntos para describir los productos y recursos para un problema particular, índices para hacer referencia a elementos de esos conjuntos y tablas de datos que contienen los valores de los parámetros relevantes.

Generalizar modelos matemáticos de esta manera es una característica común de muchas aplicaciones de ciencia de datos. A continuación veremos cómo se facilita este tipo de generalización en Pyomo.

## El modelo de producción en Pyomo

Como antes, comenzamos la construcción de un modelo Pyomo creando un `ConcreteModel`.

In [5]:
model = pyo.ConcreteModel()

En modelado y optimización matemática, un conjunto sirve como una colección indexada de elementos que le permite definir variables, restricciones y otros componentes del modelo de forma generalizada. El componente `Set()` de Pyomo tiene el mismo propósito: se utiliza para definir conjuntos de índices sobre los cuales se pueden definir variables, parámetros, restricciones u objetivos.

Usamos Pyomo `Set()` para construir conjuntos correspondientes a los productos y recursos. Cada conjunto se inicializa con las claves del diccionario para las tablas de atributos relevantes. Posteriormente se convertirán en índices de parámetros, variables de decisión y restricciones.

In [6]:
model.PRODUCTS = pyo.Set(initialize=products.keys())
model.RESOURCES = pyo.Set(initialize=resources.keys())

El siguiente paso es introducir los parámetros que se utilizarán en las restricciones y funciones objetivo. Estos están indexados por productos, recursos o ambos. Los valores de los parámetros se asignan al nombre del modelo. Usamos decoradores de Pyomo para declarar estos parámetros, donde la función entre decorados se convierte en el nombre del parámetro y la función devuelve el valor del parámetro de los conjuntos de datos del problema. Esto forma la interfaz entre los datos del problema y el modelo Pyomo.

Este paso de declarar objetos Pyomo `Param` a menudo se omite en las aplicaciones de Pyomo. Al hacerlo, el modelador elige incorporar la representación de datos externos directamente en los objetivos y restricciones del problema. Esto puede ser efectivo, mantiene el código más corto y puede eliminar cierta sobrecarga computacional. Sin embargo, también desdibuja el límite entre la representación de datos y las declaraciones del modelo. Cualquier cambio en la representación de los datos puede requerir editar cada lugar donde se utilizan esos datos en el modelo. Al definir los parámetros del modelo con `Param()`, la interfaz para la representación de datos se ubica en una parte claramente definida de un modelo más grande, lo que mejora significativamente la mantenibilidad a largo plazo de los modelos. Esta preocupación puede resultar excesiva en modelos pequeños como el que tenemos aquí, pero es una consideración clave al construir aplicaciones más complejas.

Nota: El dominio de los límites se establece en `Any` porque algunos de ellos tomarán el valor `None`. Pyomo omitirá los límites inferior o superior que tengan un valor de `None`, por lo que esta es una forma de mantener la lógica simple.

In [7]:
# parameter for bounds
@model.Param(model.PRODUCTS, domain=pyo.Any)
def demand(model, product):
    return products[product]["demand"]


@model.Param(model.RESOURCES, domain=pyo.Any)
def available(model, resource):
    return resources[resource]["available"]


# parameter with price coefficients
@model.Param(model.PRODUCTS)
def cp(model, product):
    return products[product]["price"]


@model.Param(model.RESOURCES)
def cr(model, resource):
    return resources[resource]["price"]


# process parameters: a[r,p]
@model.Param(model.RESOURCES, model.PRODUCTS)
def a(model, resource, product):
    return processes[product][resource]

Las variables de decisión, $x$ y $y$, están indexadas por el conjunto de recursos y productos, respectivamente. La indexación se especifica pasando los conjuntos relevantes como primeros argumentos a Pyomo `Var()`. Además de la indexación, siempre es una buena práctica especificar los límites conocidos y fijos de las variables. Esto se hace especificando una función (en el lenguaje de Pyomo, a veces llamada regla) que devuelve el límite para un índice determinado. Aquí usamos una función lambda de Python con dos argumentos, modelo y un índice que hace referencia a un miembro de un conjunto, para devolver una tupla con el límite inferior y superior.

In [8]:
model.x = pyo.Var(
    model.RESOURCES, bounds=lambda model, resource: (0, model.available[resource])
)
model.y = pyo.Var(
    model.PRODUCTS, bounds=lambda model, product: (0, model.demand[product])
)

El objetivo se expresa con Pyomo `quicksum` que acepta un generador Python para términos sucesivos en la suma. Aquí utilizamos los parámetros $c_p$ y $c_r$ que aparecen en la versión matemática del modelo y que fueron declarados anteriormente en la versión Pyomo del modelo.

In [9]:
model.revenue = pyo.quicksum(
    model.cp[product] * model.y[product] for product in model.PRODUCTS
)
model.cost = pyo.quicksum(
    model.cr[resource] * model.x[resource] for resource in model.RESOURCES
)


# create objective
@model.Objective(sense=pyo.maximize)
def profit(model):
    return model.revenue - model.cost

El decorador Pyomo `Constraint` acepta uno o más conjuntos como argumentos. Luego, para cada miembro de cada conjunto, la función decorada crea una restricción asociada. La creación de restricciones indexadas de esta manera es un componente esencial para modelos más complejos.

In [10]:
# create indexed constraint
@model.Constraint(model.RESOURCES)
def materials_used(model, resource):
    return (
        pyo.quicksum(
            model.a[resource, product] * model.y[product] for product in model.PRODUCTS
        )
        <= model.x[resource]
    )

El último paso es resolver el modelo e informar la solución. Aquí creamos un informe simple usando `pyo.value()` para acceder a los valores de las variables de decisión y usando los conjuntos de modelos para construir iteradores para informar el valor de las variables indexadas.

In [11]:
model.pprint()

# solve
SOLVER.solve(model)

# create a solution report
print(f"Profit = {pyo.value(model.profit)}")

print("\nProduction Report")
for product in model.PRODUCTS:
    print(f" {product}  produced =  {pyo.value(model.y[product])}")

print("\nResource Report")
for resource in model.RESOURCES:
    print(f" {resource} consumed = {pyo.value(model.x[resource])}")

3 Set Declarations
    PRODUCTS : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'U', 'V'}
    RESOURCES : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'M', 'labor A', 'labor B'}
    a_index : Size=1, Index=None, Ordered=True
        Key  : Dimen : Domain             : Size : Members
        None :     2 : RESOURCES*PRODUCTS :    6 : {('M', 'U'), ('M', 'V'), ('labor A', 'U'), ('labor A', 'V'), ('labor B', 'U'), ('labor B', 'V')}

5 Param Declarations
    a : Size=6, Index=a_index, Domain=Any, Default=None, Mutable=False
        Key              : Value
              ('M', 'U') :    10
              ('M', 'V') :     9
        ('labor A', 'U') :     1
        ('labor A', 'V') :     1
        ('labor B', 'U') :     2
        ('labor B', 'V') :     1
    available : Size=3, Index=RESOURCES, Domain=Any, Default=None, Mutable=False
    

## Para expertos en Python: creación de subclases de `ConcreteModel`

Algunos lectores de estos cuadernos pueden ser desarrolladores de Python más experimentados que deseen aplicar Pyomo en aplicaciones más especializadas basadas en datos. La siguiente celda muestra cómo la clase Pyomo `ConcreteModel()` se puede ampliar mediante subclases para crear clases de modelo especializadas. Aquí creamos una subclase llamada `ProductionModel` que acepta una representación particular de los datos del problema para producir un objeto de modelo de producción. El objeto del modelo de producción hereda todos los métodos asociados con cualquier `ConcreteModel`, como `.display()`, `.solve()` y `.pprint()`, pero se puede ampliar con métodos adicionales.

In [ ]:
class ProductionModel(pyo.ConcreteModel):
    """
    A class representing a production model using Pyomo.
    """

    def __init__(self, products, resources, processes):
        """
        Initialize ProductionModel as an instance of a ConcreteModel.

        :param products: A dictionary containing product information.
        :param resources: A dictionary containing resource information.
        :param processes: A dictionary containing process information.
        """
        super().__init__("Production Model")

        # save data in the model instance
        self.products = products
        self.resources = resources
        self.processes = processes

        # flag to monitor solution status
        self.solved = False

    def build_model(self):
        """
        Build the optimization model.
        """
        # access the model
        model = self.model()

        # create sets to index variables and constraints
        model.PRODUCTS = self.products.keys()
        model.RESOURCES = self.resources.keys()

        # decision variables
        model.x = pyo.Var(
            model.RESOURCES,
            bounds=lambda model, resource: (0, self.resources[resource]["available"]),
        )
        model.y = pyo.Var(
            model.PRODUCTS,
            bounds=lambda model, product: (0, self.products[product]["demand"]),
        )

        # use expressions to break up complex objectives
        model.revenue = pyo.quicksum(
            self.products[product]["price"] * model.y[product]
            for product in model.products
        )
        model.cost = pyo.quicksum(
            self.resources[resource]["price"] * model.x[resource]
            for resource in model.resources
        )

        # create objective
        @model.Objective(sense=pyo.maximize)
        def profit(model):
            return model.revenue - model.cost

        # create indexed constraint
        @model.Constraint(model.RESOURCES)
        def materials_used(model, resource):
            return (
                pyo.quicksum(
                    self.processes[product][resource] * model.y[product]
                    for product in model.PRODUCTS
                )
                <= model.x[resource]
            )

    def solve(self, solver=SOLVER):
        """
        Buildthe model, if necessary, then solve the optimization model.
        """
        self.build_model()
        solver.solve(self)
        self.solved = True

    def report(self):
        """
        Solve, if necessary, then report the model solution.
        """
        if not self.solved:
            self.solve(SOLVER)
        print(f"Profit = {pyo.value(self.profit)}")
        print("\nProduction Report")
        for product in self.PRODUCTS:
            print(f" {product}  produced =  {pyo.value(self.y[product])}")
        print("\nResource Report")
        for resource in self.RESOURCES:
            print(f" {resource} consumed = {pyo.value(self.x[resource])}")


m = ProductionModel(products, resources, processes)
m.report()

Profit = 2600.0

Production Report
 U  produced =  20.0
 V  produced =  60.0

Resource Report
 M consumed = 740.0
 labor A consumed = 80.0
 labor B consumed = 100.0
